# Review 1 — Regression Model Comparison

This notebook consolidates the outputs from the ten required regression notebooks.

It is **not an additional regression algorithm**. Its purpose is to create the single comparison table required by Review 1, identify the two best models by test-set R², perform 5-fold cross-validation for those models, and generate the final diagnostic plots.

## 1. Imports and Project Paths

In [ ]:
from pathlib import Path
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Locate the project root whether Jupyter is started from the repository root
# or from Review-1/notebooks.
cwd = Path.cwd().resolve()
candidates = [cwd] + list(cwd.parents)
PROJECT_ROOT = next(
    (p for p in candidates if (p / "src" / "preprocessing.py").exists()),
    cwd
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import (
    RANDOM_STATE, TARGET, load_and_engineer, split_data,
    remove_training_outliers, make_preprocessor, make_polynomial_preprocessor,
    regression_metrics
)

DATA_PATH = PROJECT_ROOT / "Review-1" / "data" / "AmesHousing.csv"
RESULTS_DIR = PROJECT_ROOT / "Review-1" / "results"
MODELS_DIR = PROJECT_ROOT / "Review-1" / "models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")

import joblib
from sklearn.model_selection import cross_val_score

## 2. Load the Dataset Using the Same Feature Engineering and Split

The comparison notebook uses the same feature engineering, `random_state=42`, 80:20 split, and training-set-only outlier treatment used by the ten model notebooks.

In [ ]:
df, X, y = load_and_engineer(DATA_PATH)
X_train, X_test, y_train, y_test = split_data(X, y)
X_train, y_train, outlier_threshold = remove_training_outliers(
    X_train, y_train, feature="Gr Liv Area", multiplier=3.0
)

print("Training shape:", X_train.shape)
print("Test shape:", X_test.shape)

## 3. Consolidated Test-Set Comparison

The ten algorithm notebooks write one CSV result each into `Review-1/results/`. The table below ranks all models by **R²**, as required by the Review 1 rubric.

In [ ]:
result_files = sorted(RESULTS_DIR.glob("*.csv"))
result_frames = []

for path in result_files:
    temp = pd.read_csv(path)
    result_frames.append(temp)

if not result_frames:
    raise FileNotFoundError(
        "No model result CSVs found. Run the ten algorithm notebooks first."
    )

comparison = pd.concat(result_frames, ignore_index=True)
comparison = comparison[["Model", "R2", "RMSE", "MAE"]].sort_values(
    "R2", ascending=False
).reset_index(drop=True)

display(comparison.style.format({
    "R2": "{:.4f}",
    "RMSE": "{:,.2f}",
    "MAE": "{:,.2f}"
}))

comparison.to_csv(RESULTS_DIR / "regression_comparison.csv", index=False)
print("Saved consolidated table to:", RESULTS_DIR / "regression_comparison.csv")

## 4. Two Best-Performing Models — 5-Fold Cross-Validated R²

The course guidelines require 5-fold cross-validation for the two best-performing models. The models are selected from the common held-out test-set ranking above.

In [ ]:
top_two = comparison.head(2)

cv_rows = []
for model_name in top_two["Model"]:
    filename = {
        "Linear Regression": "linear_regression.joblib",
        "Ridge Regression": "ridge_regression.joblib",
        "Lasso Regression": "lasso_regression.joblib",
        "ElasticNet Regression": "elasticnet_regression.joblib",
        "Polynomial Regression": "polynomial_regression.joblib",
        "Decision Tree Regressor": "decision_tree_regressor.joblib",
        "Random Forest Regressor": "random_forest_regressor.joblib",
        "Gradient Boosting Regressor": "gradient_boosting_regressor.joblib",
        "Support Vector Regressor (SVR)": "svr.joblib",
        "K-Nearest Neighbors Regressor": "knn_regressor.joblib",
    }[model_name]

    estimator = joblib.load(MODELS_DIR / filename)
    scores = cross_val_score(
        estimator,
        X_train,
        y_train,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )

    cv_rows.append({
        "Model": model_name,
        "CV_R2_Mean": scores.mean(),
        "CV_R2_Std": scores.std(),
        "Fold_1": scores[0],
        "Fold_2": scores[1],
        "Fold_3": scores[2],
        "Fold_4": scores[3],
        "Fold_5": scores[4],
    })

cv_results = pd.DataFrame(cv_rows)
display(cv_results.style.format({
    "CV_R2_Mean": "{:.4f}",
    "CV_R2_Std": "{:.4f}",
    "Fold_1": "{:.4f}",
    "Fold_2": "{:.4f}",
    "Fold_3": "{:.4f}",
    "Fold_4": "{:.4f}",
    "Fold_5": "{:.4f}",
}))

cv_results.to_csv(RESULTS_DIR / "top_two_cross_validation.csv", index=False)

## 5. Best Model — Predicted vs Actual and Residual Plot

The best model is selected by held-out test-set R². The fitted model is loaded from the corresponding algorithm notebook.

In [ ]:
best_name = comparison.iloc[0]["Model"]
model_files = {
    "Linear Regression": "linear_regression.joblib",
    "Ridge Regression": "ridge_regression.joblib",
    "Lasso Regression": "lasso_regression.joblib",
    "ElasticNet Regression": "elasticnet_regression.joblib",
    "Polynomial Regression": "polynomial_regression.joblib",
    "Decision Tree Regressor": "decision_tree_regressor.joblib",
    "Random Forest Regressor": "random_forest_regressor.joblib",
    "Gradient Boosting Regressor": "gradient_boosting_regressor.joblib",
    "Support Vector Regressor (SVR)": "svr.joblib",
    "K-Nearest Neighbors Regressor": "knn_regressor.joblib",
}
best_model = joblib.load(MODELS_DIR / model_files[best_name])
best_pred = best_model.predict(X_test)

best_metrics = regression_metrics(y_test, best_pred)
print("Best model:", best_name)
display(pd.DataFrame([best_metrics], index=[best_name]))

plt.figure(figsize=(7, 6))
sns.scatterplot(x=y_test, y=best_pred, alpha=0.65)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
plt.plot(lims, lims, linestyle="--", label="Ideal: Actual = Predicted")
plt.title(f"{best_name} — Predicted vs Actual")
plt.xlabel("Actual SalePrice")
plt.ylabel("Predicted SalePrice")
plt.legend()
plt.tight_layout()
plt.show()

residuals = y_test - best_pred
plt.figure(figsize=(8, 5))
sns.scatterplot(x=best_pred, y=residuals, alpha=0.65)
plt.axhline(0, linestyle="--", label="Zero Residual")
plt.title(f"{best_name} — Residual Plot")
plt.xlabel("Predicted SalePrice")
plt.ylabel("Residual (Actual - Predicted)")
plt.legend()
plt.tight_layout()
plt.show()

## 6. Feature Importance for a Tree-Based Model

Review 1 requires a feature-importance visualization for at least one tree-based model. The code below chooses the best-performing tree-based algorithm from the comparison table, regardless of which model wins overall.

In [ ]:
tree_names = {
    "Decision Tree Regressor",
    "Random Forest Regressor",
    "Gradient Boosting Regressor",
}
tree_candidates = comparison[comparison["Model"].isin(tree_names)]

if tree_candidates.empty:
    print("Run the tree-based notebooks first.")
else:
    tree_name = tree_candidates.iloc[0]["Model"]
    tree_model = joblib.load(MODELS_DIR / model_files[tree_name])

    preprocessor = tree_model.named_steps["preprocess"]
    estimator = tree_model.named_steps["model"]

    feature_names = preprocessor.get_feature_names_out()
    importance = pd.Series(estimator.feature_importances_, index=feature_names)
    top_importance = importance.sort_values(ascending=False).head(20)

    plt.figure(figsize=(10, 7))
    top_importance.sort_values().plot(kind="barh")
    plt.title(f"{tree_name} — Top 20 Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()

## 7. Team Interpretation

**Write the team's own final interpretation here.**

Explain:
- which model performed best and why,
- whether the test-set and cross-validation results agree,
- the practical meaning of R², RMSE, and MAE,
- which features appear important,
- limitations and possible improvements.

Do not copy generic interpretations; base the discussion on the actual outputs produced by this project.